In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from norm_manual import my_batchnorm, my_layernorm, normalized_example

torch.manual_seed(0)
x = torch.randn(16)  # один фиксированный пример

# 200 раз кладём x в батч со случайными "соседями" и смотрим,
# насколько сильно скачет его нормализованное значение.
for k in [3, 15, 63]:
    outs_bn = torch.stack([normalized_example(my_batchnorm, x, torch.randn(k, 16)) for _ in range(200)])
    outs_ln = torch.stack([normalized_example(my_layernorm, x, torch.randn(k, 16)) for _ in range(200)])
    print(f"batch = {k + 1:>3}   разброс выхода BN: {outs_bn.std(dim=0).mean():.3f}   "
          f"разброс выхода LN: {outs_ln.std(dim=0).mean():.3f}")

batch =   4   разброс выхода BN: 0.577   разброс выхода LN: 0.000
batch =  16   разброс выхода BN: 0.284   разброс выхода LN: 0.000
batch =  64   разброс выхода BN: 0.145   разброс выхода LN: 0.000


In [3]:
import torch.nn as nn

bn = nn.BatchNorm1d(16)
ln = nn.LayerNorm(16)
one = torch.randn(1, 16)  # батч из ОДНОГО примера (как при генерации по одному запросу)

bn.train()
try:
    bn(one)
except Exception as e:
    print("BN, train, батч из 1 -> ошибка:", e)

print("LN, батч из 1 -> работает, форма:", tuple(ln(one).shape))

bn.eval()
print("BN, eval, батч из 1 -> работает, форма:", tuple(bn(one).shape))

BN, train, батч из 1 -> ошибка: Expected more than 1 value per channel when training, got input size torch.Size([1, 16])
LN, батч из 1 -> работает, форма: (1, 16)
BN, eval, батч из 1 -> работает, форма: (1, 16)
